# KB vLLM + Qwen — Colab GPU inference server

Serves **Qwen2.5-Instruct** via vLLM's OpenAI-compatible API and exposes it through a
**Cloudflare quick tunnel** so the local KB ingestion worker can reach it.

JSON output is enforced **server-side** (xgrammar token masking) via
`response_format: {"type": "json_schema", ...}` — the structured-output replacement for
the GBNF grammars used by the local llama.cpp service (`model/ollama`), which stays untouched.

**How to run**
1. Runtime ▸ Change runtime type ▸ **GPU** (L4 recommended on Pro; A100 picks the 14B model)
2. Runtime ▸ **Run all**
3. Copy `TUNNEL_URL` + `API_KEY` from the banner near the bottom
4. From the repo on your machine:
   `python model/vllm/test_endpoint.py <TUNNEL_URL> --api-key <API_KEY>`

Keep this tab open — the last cell is a keep-alive loop. A new tunnel URL is minted on every run.

In [ ]:
# -- 1 - GPU check & model auto-selection --------------------------------------
import subprocess

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()
assert gpu, "No GPU — Runtime > Change runtime type > select a GPU (L4 recommended)"
print("GPU:", gpu)

gpu_name = gpu.lower()
if "a100" in gpu_name:
    MODEL, DTYPE, MAX_LEN = "Qwen/Qwen2.5-14B-Instruct-AWQ", "auto", 16384
elif "l4" in gpu_name:
    MODEL, DTYPE, MAX_LEN = "Qwen/Qwen2.5-7B-Instruct", "auto", 8192
elif "t4" in gpu_name:
    # T4 is pre-Ampere: no bf16, no FlashAttention — AWQ weights + fp16 + xformers
    MODEL, DTYPE, MAX_LEN = "Qwen/Qwen2.5-7B-Instruct-AWQ", "half", 8192
else:
    MODEL, DTYPE, MAX_LEN = "Qwen/Qwen2.5-7B-Instruct-AWQ", "auto", 8192
print(f"Selected: {MODEL}  dtype={DTYPE}  max_model_len={MAX_LEN}")

In [ ]:
# -- 2 - Install vLLM (CUDA-matched) + cloudflared (~3-6 min) ---------------------
# Plain `pip install -U vllm` breaks on Colab: the newest PyPI wheel links against
# CUDA 13 (ImportError: libcudart.so.13) while Colab's driver only supports CUDA 12.x
# (https://github.com/vllm-project/vllm/issues/43435). Fix: install the per-CUDA
# wheel from GitHub Releases that matches the driver, with torch from the matching
# PyTorch index — the install path the vLLM docs recommend for non-default CUDA.
import json, re, subprocess, urllib.request

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
drv = re.search(r"CUDA Version:\s*(\d+)\.(\d+)", smi)
assert drv, "could not read driver CUDA version from nvidia-smi"
drv_major = int(drv.group(1))
print(f"driver supports CUDA {drv.group(1)}.{drv.group(2)}")

%pip install -q uv

if drv_major >= 13:
    # driver can load CUDA 13 — let uv pick wheels matching the driver
    !uv pip install --system -q -U vllm --torch-backend=auto
else:
    rel = json.load(urllib.request.urlopen(
        "https://api.github.com/repos/vllm-project/vllm/releases/latest"))
    cu_wheels = {}   # 128 -> .../vllm-X.Y.Z+cu128-cp38-abi3-...x86_64.whl
    for asset in rel["assets"]:
        m = re.search(r"\+cu(\d{3})-cp\d+-abi3-.*x86_64\.whl$",
                      asset["browser_download_url"])
        if m:
            cu_wheels[int(m.group(1))] = asset["browser_download_url"]
    usable = sorted(v for v in cu_wheels if v // 10 == drv_major)
    assert usable, (
        f"release {rel['tag_name']} has no cu{drv_major}x wheel "
        f"(found: {sorted(cu_wheels)}) — check the vLLM release assets"
    )
    cu = usable[-1]
    wheel = cu_wheels[cu]
    print(f"installing {wheel.rsplit('/', 1)[-1]}  (torch index: cu{cu})")
    !uv pip install --system -q "{wheel}" --extra-index-url https://download.pytorch.org/whl/cu{cu}

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("install done")

In [ ]:
# -- 3 - Config ------------------------------------------------------------------
import secrets

PORT = 8000
API_KEY = "kb-" + secrets.token_urlsafe(16)   # tunnel is public internet — always require a key
# MODEL = "Qwen/Qwen2.5-7B-Instruct"          # uncomment to override auto-selection

print("API key:", API_KEY)

In [ ]:
# -- 4 - Launch vLLM (first run downloads the weights — 5-15 min) -----------------
import os, subprocess, time, urllib.request

env = os.environ.copy()
if "t4" in gpu_name:
    env["VLLM_ATTENTION_BACKEND"] = "XFORMERS"

cmd = [
    "vllm", "serve", MODEL,
    "--dtype", DTYPE,
    "--max-model-len", str(MAX_LEN),
    "--gpu-memory-utilization", "0.92",
    "--port", str(PORT),
    "--api-key", API_KEY,
    "--enable-prefix-caching",   # repeated system prompts hit the KV cache
]
print(" ".join(cmd))
vllm_log = open("vllm.log", "w")
vllm_proc = subprocess.Popen(cmd, stdout=vllm_log, stderr=subprocess.STDOUT, env=env)

deadline = time.time() + 30 * 60
while True:
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=5)
        print("vLLM is up")
        break
    except Exception:
        pass
    if vllm_proc.poll() is not None:
        print(open("vllm.log").read()[-6000:])
        raise RuntimeError("vLLM exited during startup — see log above")
    if time.time() > deadline:
        raise TimeoutError("vLLM not healthy after 30 min — check vllm.log")
    time.sleep(10)

In [ ]:
# -- 5 - Local smoke test: schema-constrained JSON --------------------------------
import json, requests

# mirror of model/vllm/schemas.py::ENTITY_INTENT_SCHEMA — keep in sync
ENTITY_INTENT_SCHEMA = {
    "type": "object",
    "properties": {
        "summary": {"type": "string"},
        "entities": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "description": {"type": "string"},
                },
                "required": ["name", "description"],
                "additionalProperties": False,
            },
        },
        "intents": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["summary", "entities", "intents"],
    "additionalProperties": False,
}

resp = requests.post(
    f"http://localhost:{PORT}/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={
        "model": MODEL,
        "messages": [
            {"role": "system", "content": (
                "Extract a one-sentence summary, named entities (each with a short "
                "description) and user intents from the text. Respond in JSON."
            )},
            {"role": "user", "content": (
                "AeroFlow's Q3 revenue grew 18% after the Singapore hub opened, "
                "said CFO Mira Chen."
            )},
        ],
        "max_tokens": 400,
        "temperature": 0.1,
        "response_format": {
            "type": "json_schema",
            "json_schema": {"name": "entity_intent", "strict": True,
                            "schema": ENTITY_INTENT_SCHEMA},
        },
    },
    timeout=120,
)
resp.raise_for_status()
out = json.loads(resp.json()["choices"][0]["message"]["content"])
print(json.dumps(out, indent=2))
assert {"summary", "entities", "intents"} <= out.keys()
print("\nschema-constrained output OK")

In [ ]:
# -- 6 - Open the Cloudflare tunnel ------------------------------------------------
import re, subprocess, time

cf_log = open("cloudflared.log", "w")
cf_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
    stdout=cf_log, stderr=subprocess.STDOUT,
)

TUNNEL_URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
    if m:
        TUNNEL_URL = m.group(0)
        break
    time.sleep(1)
assert TUNNEL_URL, "tunnel URL not found — check cloudflared.log"

print("=" * 70)
print("  TUNNEL_URL :", TUNNEL_URL)
print("  API_KEY    :", API_KEY)
print("  MODEL      :", MODEL)
print("=" * 70)
print(f'''
Test from your machine (repo root):
  python model/vllm/test_endpoint.py {TUNNEL_URL} --api-key {API_KEY}

Or curl:
  curl {TUNNEL_URL}/v1/models -H "Authorization: Bearer {API_KEY}"
''')

In [ ]:
# -- 7 - Keep-alive — leave this cell running --------------------------------------
import time

mins = 0
while True:
    time.sleep(300)
    mins += 5
    alive = "ok" if vllm_proc.poll() is None else "DEAD"
    tunnel = "ok" if cf_proc.poll() is None else "DEAD"
    print(f"[{mins:>4} min] vllm={alive}  tunnel={tunnel}  url={TUNNEL_URL}")

## Notes

- **URL churn**: quick tunnels mint a new `*.trycloudflare.com` URL each run. The worker
  reads the endpoint from an env var, so it's a one-line compose change per session.
- **Session limits**: Colab Pro runtimes last up to ~24 h and disconnect when idle.
  This is a dev/testing setup, not a production deployment.
- **Auth**: vLLM rejects any request without `Authorization: Bearer <API_KEY>` — the
  tunnel URL is public, so don't remove `--api-key`.
- **Prefix caching** replaces the custom X-System-Type KV-snapshot machinery from the
  llama.cpp service: identical system prompts are cached automatically.